In [ ]:
# ---------------- IMPORTS & LOGGING SETUP ----------------
import pandas as pd
import numpy as np
import sqlite3
import logging
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Logging configuration
logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Notebook pipeline started")

print("Imports and logging initialized ✅")

In [ ]:
# ---------------- LOAD RAW DATA ----------------
df_raw = pd.read_csv("data.csv")

logging.info("Raw data loaded successfully")

print("Raw dataset shape:", df_raw.shape)
df_raw.head()

In [ ]:
# ---------------- STORE RAW DATA IN DATABASE ----------------
conn = sqlite3.connect("housing_engineered.db")

df_raw.to_sql("raw_housing_data", conn, if_exists="replace", index=False)

logging.info("Raw data stored in SQLite database (raw_housing_data table)")

print("Raw data stored in database ✅")

In [ ]:
# ---------------- CREATE WORKING COPY ----------------
df = df_raw.copy()

logging.info("Working copy of dataset created")

df.head()

In [ ]:
# ---------------- DATA CLEANING: HANDLE MISSING VALUES ----------------
missing_before = df["total_bedrooms"].isnull().sum()

df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

missing_after = df["total_bedrooms"].isnull().sum()

logging.info(f"Missing values handled in total_bedrooms. Before: {missing_before}, After: {missing_after}")

print("Missing values handled successfully ✅")
df.isnull().sum()

In [ ]:
# ---------------- FEATURE ENGINEERING ----------------
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

logging.info("Feature engineering completed (3 new features added)")

print("Feature engineering done ✅")
df.head()

In [ ]:
# ---------------- CATEGORICAL ENCODING ----------------
df = pd.get_dummies(df, columns=["ocean_proximity"])

logging.info("Categorical encoding completed using one-hot encoding")

print("Categorical encoding done ✅")
df.head()

In [ ]:
# ---------------- BOOLEAN TO INTEGER CONVERSION ----------------
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

logging.info(f"Converted boolean columns to int: {list(bool_cols)}")

df.info()

In [ ]:
# ---------------- DATA VALIDATION ----------------
logging.info("Starting data validation checks")

# Check nulls
total_nulls = df.isnull().sum().sum()

# Check duplicates
total_duplicates = df.duplicated().sum()

assert total_nulls == 0, "Null values still exist!"
assert total_duplicates == 0, "Duplicate rows detected!"
assert (df["population"] >= 0).all(), "Negative population found!"

logging.info("Data validation passed successfully")

print("Data validation passed ✅")
print("Null values:", total_nulls)
print("Duplicates:", total_duplicates)

In [ ]:
# ---------------- FEATURE SCALING (EXCLUDE TARGET) ----------------
target_col = "median_house_value"

# Separate features and target
X_features = df.drop(columns=[target_col])
y_target = df[target_col]

# Select only numeric feature columns
numeric_feature_cols = X_features.select_dtypes(include=np.number).columns

# Apply scaling only to features
scaler = StandardScaler()
X_features[numeric_feature_cols] = scaler.fit_transform(X_features[numeric_feature_cols])

# Recombine scaled features + original target
df = pd.concat([X_features, y_target], axis=1)

logging.info("Feature scaling completed (target variable excluded)")

print("Scaling completed (target preserved) ✅")
print(df["median_house_value"].head())  # should show real values

In [ ]:
# ---------------- STORE PROCESSED DATA IN DATABASE ----------------
df.to_sql("processed_housing_data", conn, if_exists="replace", index=False)

logging.info("Processed data stored in SQLite (processed_housing_data table)")

print("Processed data stored in database ✅")

In [ ]:
# ---------------- EXPORT PROCESSED CSV ----------------
df.to_csv("processed_housing_data.csv", index=False)

logging.info("Processed CSV exported successfully")

print("Processed CSV exported: processed_housing_data.csv ✅")

In [ ]:
# ---------------- PREPARE DATA FOR ML ----------------
logging.info("Preparing dataset for model training")

X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

logging.info(f"Feature shape: {X.shape}, Target shape: {y.shape}")

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# ---------------- TRAIN-TEST SPLIT ----------------
logging.info("Performing train-test split")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

logging.info(f"Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# ---------------- LINEAR REGRESSION TRAINING ----------------
logging.info("Training Linear Regression model")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

logging.info("Linear Regression training completed")

print("Linear Regression model trained ✅")

In [ ]:
# ---------------- LINEAR REGRESSION EVALUATION ----------------
logging.info("Evaluating Linear Regression model")

y_pred_lr = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

logging.info(f"Linear Regression - MAE: {mae_lr}, RMSE: {rmse_lr}, R2: {r2_lr}")

print("Linear Regression Performance:")
print("MAE:", round(mae_lr, 2))
print("RMSE:", round(rmse_lr, 2))
print("R2 Score:", round(r2_lr, 4))

In [ ]:
# ---------------- RANDOM FOREST TRAINING ----------------
logging.info("Training Random Forest Regressor")

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

logging.info("Random Forest training completed")

print("Random Forest model trained 🚀")

In [ ]:
# ---------------- RANDOM FOREST EVALUATION ----------------
logging.info("Evaluating Random Forest model")

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

logging.info(f"Random Forest - MAE: {mae_rf}, RMSE: {rmse_rf}, R2: {r2_rf}")

print("Random Forest Performance:")
print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))
print("R2 Score:", round(r2_rf, 4))

In [ ]:
# ---------------- MODEL COMPARISON ----------------
logging.info("Comparing model performances")

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R2 Score": [r2_lr, r2_rf]
})

logging.info("Model comparison completed")

comparison

In [ ]:
# ---------------- SAVE BEST MODEL ----------------
logging.info("Saving final trained model")

joblib.dump(rf_model, "house_price_model.pkl")

logging.info("Model saved as house_price_model.pkl")

print("Final model saved successfully 💾")

In [ ]:
# ---------------- CLOSE DATABASE CONNECTION ----------------
conn.close()

logging.info("Database connection closed. Pipeline completed successfully")

print("End-to-End Data Engineering + ML Pipeline Completed 🎉")

In [ ]:
# ---------------- CHECK FEATURE ORDER ----------------
logging.info("Checking feature order for prediction")

print("Feature columns used in model:")
for i, col in enumerate(X.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# ---------------- CREATE SAMPLE INPUT FOR PREDICTION ----------------
logging.info("Creating sample input for model prediction")

sample_data = {
    'longitude': -1.2,
    'latitude': 1.0,
    'housing_median_age': 0.5,
    'total_rooms': 0.2,
    'total_bedrooms': 0.1,
    'population': -0.3,
    'households': -0.2,
    'median_income': 1.5,  # High income area (important)
    'rooms_per_household': 0.4,
    'bedrooms_per_room': -0.2,
    'population_per_household': -0.1,
    'ocean_proximity_<1H OCEAN': 0,
    'ocean_proximity_INLAND': 0,
    'ocean_proximity_ISLAND': 0,
    'ocean_proximity_NEAR BAY': 0,
    'ocean_proximity_NEAR OCEAN': 1  # Premium location
}

# Convert to DataFrame (MUST match training format)
sample_df = pd.DataFrame([sample_data])

logging.info("Sample input dataframe created for prediction")

sample_df

In [ ]:
# ---------------- LOAD TRAINED MODEL ----------------
import joblib

logging.info("Loading saved model for prediction")

model = joblib.load("house_price_model.pkl")

logging.info("Model loaded successfully")

print("Model loaded successfully ✅")

In [ ]:
# ---------------- MAKE PREDICTION ----------------
logging.info("Starting house price prediction")

predicted_price = model.predict(sample_df)

logging.info(f"Prediction completed. Predicted value: {predicted_price[0]}")

print("🏠 Predicted House Price: $", round(predicted_price[0], 2))

In [ ]:
# ---------------- SAVE SCALER FOR PRODUCTION ----------------


joblib.dump(scaler, "scaler.pkl")

logging.info("Scaler saved for FastAPI + Streamlit inference pipeline")
print("Scaler saved successfully as scaler.pkl ✅")